## **SECTION A: INTRO**

#### **CONTEXT & BIG IDEA**

* **WHO:** **Travelers** choosing **waterfront resorts** in Kenya.
* **WHAT:** Understand what drives **customer satisfaction**, and how **noise** impacts ratings differently at the coast vs. inland lakes.
* **HOW:** By analyzing **168 resorts** using the **Google Places API** (ratings, review text, and noise sources within 1 km).

* **BIG IDEA:** Travelers should choose resorts based on the **vibe they want**. At the **coast**, noise doesn't hurt ratings, but at **inland lakes**, noisier resorts get significantly **lower ratings**.

#### **THE DATASET**

* **What one row is:** One **waterfront resort** in Kenya (168 in total).
* **Water body type:** **Marine coast**, **inland lake**, or **safari river**.
* **Rating:** The resort's **Google guest rating**.
* **Noise counts:** How many **bars, nightclubs, transit stations and worship places** are within 1 km.
* **Review mentions:** Which topics (**service, food, noise**, etc.) guests talk about in their reviews.

## **SECTION B: SETUP**

**Setup:** Importing the libraries, loading the Google API key from a `.env` file, and starting the Google Maps client. Re-running the Google cells needs your own API key.

In [2]:
import os
import pandas as pd
import googlemaps
from dotenv import load_dotenv

# Loading secure environment variables from .env file
load_dotenv()
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

if not API_KEY:
    raise ValueError("❌ Error: GOOGLE_MAPS_API_KEY not found in your .env file!")

# Initialize the Google Maps client
gmaps = googlemaps.Client(key=API_KEY)
print("✅ Environment initialized and Google Maps client ready.")

✅ Environment initialized and Google Maps client ready.


*The Google cells (Sections C to F) need an API key and take a while to run. The analysis can be run from Section G using the saved dataset. That tells any reader how to run it.*

### **SECTION C: FINDING THE RESORTS**

**Finding the resorts:** Ten Google Places searches cover the coast, inland lakes and safari rivers across Kenya. Each resort is kept once, labelled by the search that found it, and stored with its name, address, rating and coordinates.

In [19]:
# Search queries covering coastal marine waters and inland water bodies (Lakes & Rivers)
expanded_water_search_queries = [
    # Coastal Marine Waters
    "resorts in Diani Beach Kenya",
    "beach resorts in Mombasa Shanzu Nyali",
    "resorts in Watamu and Malindi Kenya",
    "hotels in Lamu Archipelago Shela",
    # Inland Lakes (Rift Valley & Western Kenya)
    "resorts around Lake Naivasha Kenya",
    "hotels near Lake Nakuru Kenya",
    "lodges around Lake Elementaita",
    "waterfront hotels in Kisumu Lake Victoria",
    # Major Safari River Lodges
    "lodges along Mara River Masai Mara",
    "river lodges Samburu Ewaso Nyiro"
]

all_water_resorts_dict = {}

print("🌍 Scanning and pulling water-proximate resorts across Kenya (Coastal & Inland)...")

for query in expanded_water_search_queries:
    print(f"🔍 Executing search query: '{query}'...")
    try:
        response = gmaps.places(query=query)
        results = response.get("results", [])
        
        for place in results:
            name = place.get("name")
            place_id = place.get("place_id")
            
            if place_id and place_id not in all_water_resorts_dict:
                address = place.get("formatted_address", "")
                rating = place.get("rating", None)
                lat = place.get("geometry", {}).get("location", {}).get("lat", None)
                lng = place.get("geometry", {}).get("location", {}).get("lng", None)
                
                # Dynamic Region / Water Body Categorization
                region = "Other Water Body"
                q_lower = query.lower()
                if "diani" in q_lower or "galu" in q_lower:
                    region = "Coast - Diani & South"
                elif "mombasa" in q_lower or "shanzu" in q_lower or "nyali" in q_lower:
                    region = "Coast - Mombasa & North"
                elif "watamu" in q_lower or "malindi" in q_lower:
                    region = "Coast - Watamu & Malindi"
                elif "lamu" in q_lower or "shela" in q_lower:
                    region = "Coast - Lamu"
                elif "naivasha" in q_lower:
                    region = "Inland Lake - Naivasha"
                elif "nakuru" in q_lower:
                    region = "Inland Lake - Nakuru"
                elif "elementaita" in q_lower:
                    region = "Inland Lake - Elementaita"
                elif "kisumu" in q_lower or "victoria" in q_lower:
                    region = "Inland Lake - Victoria"
                elif "mara" in q_lower or "samburu" in q_lower or "river" in q_lower:
                    region = "Safari River Lodge"

                all_water_resorts_dict[place_id] = {
                    "ResortName": name,
                    "Address": address,
                    "WaterBodyType": "Marine Coast" if "Coast" in region else ("Inland Lake" if "Lake" in region else "Safari River"),
                    "Location": region,
                    "Latitude": lat,
                    "Longitude": lng,
                    "GoogleRating": rating,
                    "BeachCrowdIndex": None,
                    "QuietnessScore": None,
                    "PrivateBeachAccess": None,
                    "SatisfactionPercent": None
                }
    except Exception as e:
        print(f"⚠️ Error with query '{query}': {e}")

# Converting to DataFrame
df_resorts = pd.DataFrame(list(all_water_resorts_dict.values()))
print(f"\n🚀 Successfully compiled {len(df_resorts)} water-proximate properties across Kenya.")
df_resorts.head()

🌍 Scanning and pulling water-proximate resorts across Kenya (Coastal & Inland)...
🔍 Executing search query: 'resorts in Diani Beach Kenya'...
🔍 Executing search query: 'beach resorts in Mombasa Shanzu Nyali'...
🔍 Executing search query: 'resorts in Watamu and Malindi Kenya'...
🔍 Executing search query: 'hotels in Lamu Archipelago Shela'...
🔍 Executing search query: 'resorts around Lake Naivasha Kenya'...
🔍 Executing search query: 'hotels near Lake Nakuru Kenya'...
🔍 Executing search query: 'lodges around Lake Elementaita'...
🔍 Executing search query: 'waterfront hotels in Kisumu Lake Victoria'...
🔍 Executing search query: 'lodges along Mara River Masai Mara'...
🔍 Executing search query: 'river lodges Samburu Ewaso Nyiro'...

🚀 Successfully compiled 169 water-proximate properties across Kenya.


,ResortName,Address,WaterBodyType,Location,Latitude,Longitude,GoogleRating,BeachCrowdIndex,QuietnessScore,PrivateBeachAccess,SatisfactionPercent
0,Diani Reef Beach Resort & Spa,"Diani Beach Road, Diani Beach, Kenya",Marine Coast,Coast - Diani & South,-4.278903,39.593257,4.3,None,None,None,None
1,Swahili Beach Resort,"Ukunda, Diani Beach Road, Kwale, Kenya",Marine Coast,Coast - Diani & South,-4.292183,39.586675,4.4,None,None,None,None
2,Diamonds Leisure Beach & Golf Resort,"Diani Road, South, Coast, Kwale, Kenya",Marine Coast,Coast - Diani & South,-4.281405,39.592070,4.4,None,None,None,None
3,Coral Beach Resort,"Diani Beach, Kenya",Marine Coast,Coast - Diani & South,-4.289194,39.589740,4.1,None,None,None,None
4,Southern Palms Beach Resort,"Plot LR 13430, Diani Beach Road Diani, Ukunda,...",Marine Coast,Coast - Diani & South,-4.268264,39.597005,4.3,None,None,None,None


### **SECTION D: FIRST CLEANING STEP**

**Cleaning:** One search returned a different "Mara River Lodge", in Nairobi and not in the Masai Mara. It is removed by its address.

In [20]:
# Remove the wrong "Mara River Lodge" (the one in Nairobi, not the Masai Mara)
bad_address = "Emabakasi,Airport North Road Behind Rubis Petrol Station, MWH3+899, Nairobi, Kenya"

# Keep every row whose Address is NOT the bad address
df_resorts = df_resorts[df_resorts["Address"] != bad_address]

# Print the new row count to confirm one row was removed
print(len(df_resorts))

168


### **SECTION E: COUNTING NOISE**

**Counting noise:** For each resort, count the bars, nightclubs, transit stations and worship places within 1 km. These add up to a total noise score. Google returns at most 20 results per search, so a single count cannot go above 20.

*Note: The cell below was manually interrupted (KeyboardInterrupt) because the Google Places API calls take several minutes to process. The data collected up to this point was successfully saved.*

In [21]:
def assess_detailed_noise_sources(lat, lng):
    """
    Scans a 1km radius around resort coordinates and returns individual 
    counts for specific noise/disruptor categories.
    """
    if pd.isna(lat) or pd.isna(lng):
        return {"BarsCount": 0, "NightClubsCount": 0, "TransitCount": 0, "WorshipCount": 0}
        
    noise_categories = {
        "BarsCount": "bar",
        "NightClubsCount": "night_club",
        "TransitCount": "transit_station",
        "WorshipCount": "church"
    }
    
    counts = {}
    for col_name, place_type in noise_categories.items():
        try:
            results = gmaps.places_nearby(
                location=(lat, lng),
                radius=1000, # 1km radius
                type=place_type
            )
            counts[col_name] = len(results.get("results", []))
        except Exception:
            counts[col_name] = 0
            
    # Checking mosques separately and add to worship count
    try:
        mosques = gmaps.places_nearby(location=(lat, lng), radius=1000, keyword="mosque")
        counts["WorshipCount"] += len(mosques.get("results", []))
    except Exception:
        pass
        
    return counts

print("🎧 Scanning detailed noise source categories for expanded properties...")

# Applying the function and expanding the dictionary into separate columns
noise_df_expanded = df_resorts.apply(
    lambda row: pd.Series(assess_detailed_noise_sources(row['Latitude'], row['Longitude'])), 
    axis=1
)

# Joining the new category columns back into your main DataFrame
df_resorts = pd.concat([df_resorts, noise_df_expanded], axis=1)

# Creating total combined noise score column
df_resorts['TotalNoiseSources'] = (
    df_resorts['BarsCount'] + 
    df_resorts['NightClubsCount'] + 
    df_resorts['TransitCount'] + 
    df_resorts['WorshipCount']
)

# Dropping any columns that are completely empty
df_resorts = df_resorts.dropna(axis=1, how='all')

df_resorts[['ResortName', 'WaterBodyType', 'BarsCount', 'NightClubsCount', 'TransitCount', 'WorshipCount', 'TotalNoiseSources']].head()

🎧 Scanning detailed noise source categories for expanded properties...


KeyboardInterrupt: 

### **SECTION F: THE PROBLEM AND THE FIX**

**Checking the worship count:** Testing the worship searches on one safari lodge showed that the mosque search ignores the 1 km limit and returns mosques from far away.

In [ ]:
# Test: what do the worship searches actually return for one safari lodge?
# Row 147 is Mara River Lodge in Lemek (the Masai Mara one)
lat, lng = df_resorts.loc[147, "Latitude"], df_resorts.loc[147, "Longitude"]

# Search 1: places typed as "church" within 1 km
churches = gmaps.places_nearby(location=(lat, lng), radius=1000, type="church")

# Search 2: places matching the keyword "mosque" within 1 km
mosques = gmaps.places_nearby(location=(lat, lng), radius=1000, keyword="mosque")

# Print how many each search returned, plus the first 5 names
print("Churches:", len(churches["results"]))
for p in churches["results"][:5]:
    print("  ", p["name"])

print("Mosques:", len(mosques["results"]))
for p in mosques["results"][:5]:
    print("  ", p["name"])

Churches: 0
Mosques: 20
   Almoez Mosque - Mara Rianta Kenya
   Almoez Academy
   Al Masjid-e-Nabawi
   Parklands Mosque مسجد
   The Prophet's Mosque


**Distance helper:** A function that measures the distance in metres between two points, so places outside 1 km can be thrown out.

In [2]:
# Helper: distance in metres between two lat/lng points (haversine formula)
from math import radians, sin, cos, asin, sqrt

def distance_m(lat1, lng1, lat2, lng2):
    R = 6371000  # Earth's radius in metres
    p1, p2 = radians(lat1), radians(lat2)
    dp = p2 - p1
    dl = radians(lng2 - lng1)
    h = sin(dp / 2) ** 2 + cos(p1) * cos(p2) * sin(dl / 2) ** 2
    return 2 * R * asin(sqrt(h))

# Quick test: Mara River Lodge (Lemek) to the Nairobi row we removed
print(round(distance_m(-1.140239, 35.140843, -1.321725, 36.903493) / 1000, 1), "km")

197.0 km


**Corrected worship count:** Runs the church and mosque searches again, but keeps only places truly within 1 km, and counts each place once.

In [ ]:
# Count worship places that are really within 1 km of a resort
def count_worship_nearby(lat, lng, radius=1000):
    if pd.isna(lat) or pd.isna(lng):
        return 0

    seen = set()  # place_ids already counted, so nothing is counted twice
    searches = [
        {"type": "church"},      # search 1: places typed as church
        {"keyword": "mosque"},   # search 2: places matching "mosque"
    ]

    for s in searches:
        try:
            results = gmaps.places_nearby(location=(lat, lng), radius=radius, **s)
            for p in results.get("results", []):
                loc = p["geometry"]["location"]
                # Keep it only if it is truly within the radius
                if distance_m(lat, lng, loc["lat"], loc["lng"]) <= radius:
                    seen.add(p["place_id"])
        except Exception:
            pass

    return len(seen)

# Test on Mara River Lodge (Lemek), row 147
lat, lng = df_resorts.loc[147, "Latitude"], df_resorts.loc[147, "Longitude"]
print(count_worship_nearby(lat, lng))

0


**Recounting and saving:** Applies the corrected count to every resort, rebuilds the total noise score, and saves the master dataset. Worship counts now range from 0 to 31, with an average of about 6.

*Note: The cell below was manually interrupted (KeyboardInterrupt) because the Google Places API calls take several minutes to process. The data collected up to this point was successfully saved.*

In [ ]:
# Recount worship places for every resort, using only places truly within 1 km
print("Recounting worship places (this takes a few minutes)...")

df_resorts["WorshipCount"] = df_resorts.apply(
    lambda row: count_worship_nearby(row["Latitude"], row["Longitude"]),
    axis=1
)

# Rebuild the total noise score from the four categories
df_resorts["TotalNoiseSources"] = (
    df_resorts["BarsCount"]
    + df_resorts["NightClubsCount"]
    + df_resorts["TransitCount"]
    + df_resorts["WorshipCount"]
)

# Save to a NEW file so the original stays safe
df_resorts.to_csv("Data/kenya_resorts_master.csv", index=False)

# Quick check: summary of the corrected worship counts
print(df_resorts["WorshipCount"].describe())
print(df_resorts.groupby("WaterBodyType")["WorshipCount"].mean().round(1))

Recounting worship places (this takes a few minutes)...


KeyboardInterrupt: 

### **SECTION F2: ADDING REVIEW DATA**

**Testing the extra Google data on one resort:** Before collecting it for all 168 resorts, I test on one. The cell finds the resort's Google place_id, checks it is the same place by distance, then asks for its rating, number of reviews, price level and reviews.

In [3]:
# Load the master dataset so this cell works on its own
df = pd.read_csv("Data/kenya_resorts_master.csv")

# Pick one resort to test on
row = df[df["ResortName"] == "Mara River Lodge"].iloc[0]

# Step 1: find the resort's place_id using its name and stored coordinates
found = gmaps.find_place(
    input=row["ResortName"],
    input_type="textquery",
    fields=["place_id", "name", "geometry"],
    location_bias=f"point:{row['Latitude']},{row['Longitude']}"
)
candidate = found["candidates"][0]
print(candidate["name"], candidate["place_id"])

# Step 2: check it is the same place (distance from the stored coordinates)
loc = candidate["geometry"]["location"]
print(round(distance_m(row["Latitude"], row["Longitude"], loc["lat"], loc["lng"])), "metres away")

# Step 3: ask Google for the details
details = gmaps.place(
    place_id=candidate["place_id"],
    fields=["name", "rating", "user_ratings_total", "price_level", "reviews"]
)["result"]

# Print: rating, number of reviews, price level, how many review texts came back
print(details.get("rating"), details.get("user_ratings_total"), details.get("price_level"), len(details.get("reviews", [])))

Mara River Lodge ChIJ9YdN9aZ6LBgRHLPxpMUm2so
0 metres away
4.1 135 None 5


**Collecting extra Google data for all 168 resorts:** For each resort, the cell finds its place_id, checks the distance, and collects the review count, price level and up to 5 review texts. These are added as new columns to the master dataset. A backup of the current master file is saved first.

In [4]:
# Function: get extra Google data for one resort
def get_extra_data(row):
    try:
        # Find the resort's place_id from its name and stored coordinates
        found = gmaps.find_place(
            input=row["ResortName"],
            input_type="textquery",
            fields=["place_id", "geometry"],
            location_bias=f"point:{row['Latitude']},{row['Longitude']}"
        )
        cand = found["candidates"][0]
        loc = cand["geometry"]["location"]

        # Distance check: is it the same place?
        dist = round(distance_m(row["Latitude"], row["Longitude"], loc["lat"], loc["lng"]))

        # Ask Google for the details
        d = gmaps.place(
            place_id=cand["place_id"],
            fields=["rating", "user_ratings_total", "price_level", "reviews"]
        )["result"]

        return {"PlaceID": cand["place_id"], "DistanceCheck_m": dist,
                "NewRating": d.get("rating"), "ReviewCount": d.get("user_ratings_total"),
                "PriceLevel": d.get("price_level"), "ReviewTexts": len(d.get("reviews", []))}
    except Exception:
        # If anything fails for a resort, return empty values instead of stopping
        return {"PlaceID": None, "DistanceCheck_m": None, "NewRating": None,
                "ReviewCount": None, "PriceLevel": None, "ReviewTexts": 0}

# Try it on a random sample of 10 resorts (random_state makes it repeatable)
sample = df.sample(10, random_state=1)
test = sample.apply(lambda r: pd.Series(get_extra_data(r)), axis=1)

# Show the results next to the resort names
print(pd.concat([sample[["ResortName", "WaterBodyType", "GoogleRating"]], test], axis=1).to_string())

                           ResortName WaterBodyType  GoogleRating                      PlaceID  DistanceCheck_m  NewRating  ReviewCount PriceLevel  ReviewTexts
40            7 Islands Resort Watamu  Marine Coast           4.3  ChIJCUANv0iBFRgRoJo-llFaRO0                0        4.3          718       None            5
132     Sarova Imperial Hotel, Kisumu   Inland Lake           4.2  ChIJwyVhO_WlKhgRw8d_Mhe7Gvw                0        4.2         1231       None            5
4         Southern Palms Beach Resort  Marine Coast           4.3  ChIJNeN4-kRGQBgRVxlknZWsFL8                0        4.3         3306       None            5
148  Sand River Masai Mara by Elewana  Safari River           4.8  ChIJnQfcoy8GLRgRucDgp9aNeGA                0        4.8          422       None            5
116              Brownie's Guesthouse   Inland Lake           4.3  ChIJTyRcdtSQKRgRCJu5roLoAPs                0        4.3          178       None            5
11             Soroi Blue Diani Beach  M

**Testing on 10 resorts:** A function that gets the extra Google data for one resort, tried on a random sample of 10. This shows how many resorts have a price level and review texts before running all 168.

In [7]:
# Safety first: save a backup copy of the current master dataset
df.to_csv("Data/old_files/kenya_resorts_master_v1.csv", index=False)

# Function: get extra Google data for one resort
def get_extra_data(row):
    try:
        found = gmaps.find_place(
            input=row["ResortName"],
            input_type="textquery",
            fields=["place_id", "geometry"],
            location_bias=f"point:{row['Latitude']},{row['Longitude']}"
        )
        cand = found["candidates"][0]
        loc = cand["geometry"]["location"]
        dist = round(distance_m(row["Latitude"], row["Longitude"], loc["lat"], loc["lng"]))

        d = gmaps.place(
            place_id=cand["place_id"],
            fields=["user_ratings_total", "price_level", "reviews"]
        )["result"]

        # Join the review texts into one cell, separated by " || "
        texts = " || ".join(r.get("text", "") for r in d.get("reviews", []))

        return {"PlaceID": cand["place_id"], "DistanceCheck_m": dist,
                "ReviewCount": d.get("user_ratings_total"),
                "PriceLevel": d.get("price_level"), "ReviewText": texts}
    except Exception:
        # If anything fails for a resort, store blanks and carry on
        return {"PlaceID": None, "DistanceCheck_m": None,
                "ReviewCount": None, "PriceLevel": None, "ReviewText": ""}

print("Collecting extra data for all resorts (this takes a few minutes)...")
extra = df.apply(lambda r: pd.Series(get_extra_data(r)), axis=1)

# Add the new columns to the main table
df = pd.concat([df, extra], axis=1)

# Save back to the master file (still one dataset)
df.to_csv("Data/kenya_resorts_master.csv", index=False)

# Quick check of what we got
print(df.shape)
print("Missing review counts:", df["ReviewCount"].isna().sum())
print("Resorts with a price level:", df["PriceLevel"].notna().sum())
print("Furthest distance check (metres):", df["DistanceCheck_m"].max())

(168, 22)
Missing review counts: ReviewCount    1
ReviewCount    1
dtype: int64
Resorts with a price level: PriceLevel    0
PriceLevel    0
dtype: int64
Furthest distance check (metres): DistanceCheck_m    12245
DistanceCheck_m    12245
dtype: int64


**Checking for repeated columns:** After adding the new Google data, the table has more columns than expected. This cell shows which column names appear more than once.

In [8]:
# Show which column names appear more than once
dupes = df.columns[df.columns.duplicated()].tolist()
print("Repeated columns:", dupes)

# Show every column name, in order
print(list(df.columns))

Repeated columns: ['PlaceID', 'DistanceCheck_m', 'ReviewCount', 'PriceLevel', 'ReviewText']
['ResortName', 'Address', 'WaterBodyType', 'Location', 'Latitude', 'Longitude', 'GoogleRating', 'BarsCount', 'NightClubsCount', 'TransitCount', 'WorshipCount', 'TotalNoiseSources', 'PlaceID', 'DistanceCheck_m', 'ReviewCount', 'PriceLevel', 'ReviewText', 'PlaceID', 'DistanceCheck_m', 'ReviewCount', 'PriceLevel', 'ReviewText']


**Removing the repeated columns:** The new Google columns were added twice. This keeps the first copy of each column and saves the cleaned master dataset.

In [9]:
# Keep only the first copy of each column
# duplicated() marks 2nd and later copies as True; the ~ flips it so we keep the first
df = df.loc[:, ~df.columns.duplicated()]

# Save the cleaned master dataset
df.to_csv("Data/kenya_resorts_master.csv", index=False)

# Check: should print (168, 17) and then False
print(df.shape)
print(df.columns.duplicated().any())

(168, 17)
False


**Checking the place matches:** Showing resorts where the Google match is more than 500 metres from the stored coordinates. These may be a different place with a similar name, so their review data cannot be trusted until checked.

In [10]:
# Resorts where the matched place is more than 500 m from the stored coordinates
far = df[df["DistanceCheck_m"] > 500]

# Show how many, then list them from furthest to closest
print("Resorts with a far match:", len(far))
print(far[["ResortName", "Address", "WaterBodyType", "DistanceCheck_m"]]
      .sort_values("DistanceCheck_m", ascending=False)
      .to_string())

# Also show resorts where no match was found at all
print("No match found:", df["PlaceID"].isna().sum())

Resorts with a far match: 1
            ResortName                   Address WaterBodyType  DistanceCheck_m
125  Elementaita lodge  Mbaruk Rd, Nakuru, Kenya   Inland Lake            12245
No match found: 0


**Handling the wrong match:** "Elementaita lodge" was matched to a place 12 km away, so its new Google data (place_id, review count, review text) is blanked. The resort stays in the dataset with its original rating.

In [11]:
# Blank the new Google data for any resort whose match is more than 500 m away
bad = df["DistanceCheck_m"] > 500
df.loc[bad, ["PlaceID", "ReviewCount", "PriceLevel", "ReviewText"]] = None

# Save the master dataset
df.to_csv("Data/kenya_resorts_master.csv", index=False)

# Check: resorts that still have no review count (and why)
print(df[df["ReviewCount"].isna()][["ResortName", "WaterBodyType", "GoogleRating"]].to_string())

            ResortName WaterBodyType  GoogleRating
125  Elementaita lodge   Inland Lake           5.0
126         HAPO LODGE   Inland Lake           NaN


**Experience: what guests talk about:** For each resort, check whether its reviews mention each topic (service, food, cleanliness, noise, setting, rooms, price). This uses keywords, so it counts mentions only, not whether they are positive or negative. Google returns about 5 reviews per resort, so this is a sample.

In [12]:
# Topics, and the words that signal each one
topics = {
    "Service": ["staff", "service", "friendly", "helpful", "hospitality"],
    "Food": ["food", "breakfast", "dinner", "lunch", "restaurant", "buffet", "meal"],
    "Cleanliness": ["clean", "dirty", "hygiene", "spotless"],
    "Noise": ["noise", "noisy", "loud", "quiet", "peaceful", "music"],
    "Setting & views": ["view", "scenery", "beach", "lake", "river", "garden"],
    "Rooms": ["room", "bed", "bathroom", "shower"],
    "Price": ["price", "expensive", "overpriced", "value", "cheap"],
}

# All review text in lower case (blank reviews become empty text)
text = df["ReviewText"].fillna("").str.lower()

# For each topic: True if any of its words appears at the START of a word
# (\b means "start of a word", so "view" does not match "review")
for topic, words in topics.items():
    pattern = "|".join(r"\b" + w for w in words)
    df["Mentions_" + topic] = text.str.contains(pattern)

# Count how many resorts mention each topic
mention_cols = [c for c in df.columns if c.startswith("Mentions_")]
print(df[mention_cols].sum().sort_values(ascending=False))
print("Resorts with review text:", (text != "").sum())

Mentions_Service            159
Mentions_Food               148
Mentions_Rooms              145
Mentions_Setting & views    129
Mentions_Cleanliness        112
Mentions_Noise               94
Mentions_Price               66
dtype: int64
Resorts with review text: 166


**Saving the topic columns:** Adding the yes/no topic columns to the master dataset.

In [13]:
# Save the master dataset with the new topic columns
df.to_csv("Data/kenya_resorts_master.csv", index=False)

# Check: rows and columns, and that no column name repeats (expect 24 columns and False)
print(df.shape)
print(df.columns.duplicated().any())

(168, 24)
False


**Making the dataset Tableau-friendly:** Review texts contain line breaks, which Tableau reads as new rows. This replaces them with spaces and saves the master dataset again.

In [15]:
# Replace line breaks inside review text with a single space
df["ReviewText"] = df["ReviewText"].fillna("").str.replace(r"[\r\n]+", " ", regex=True)

# Save the master dataset again
df.to_csv("Data/kenya_resorts_master.csv", index=False)

# Check: should print (168, 24) and then 0 (no line breaks left)
print(df.shape)
print(df["ReviewText"].str.contains("\n").sum())

(168, 24)
0


**Dropping the empty price column:** Google returned no price level for any resort, so the column is empty. It is removed, and the Value dimension is replaced by Destination.

In [16]:
# Drop the empty PriceLevel column
df = df.drop(columns=["PriceLevel"])

# Save the master dataset again
df.to_csv("Data/kenya_resorts_master.csv", index=False)

# Check: should print (168, 23) and then False
print(df.shape)
print(df.columns.duplicated().any())

(168, 23)
False


**Reputation groups:** Among resorts with at least 50 reviews, label the 10 highest rated "Top 5", the 10 lowest rated "Bottom 5", and the rest "Middle". Ties are broken by review count.

In [9]:
import pandas as pd

df = pd.read_csv("Data/kenya_resorts_final.csv")
df["ReputationGroup"] = "Middle"

# Loop through each resort type and rank them separately
for resort_type in df["WaterBodyType"].unique():
    # Get resorts of this type with at least 50 reviews
    group = df[(df["WaterBodyType"] == resort_type) & (df["ReviewCount"] >= 50)].sort_values(
        ["GoogleRating", "ReviewCount"], ascending=[False, False])
    
    if len(group) >= 10:
        # Top 5 for this type
        df.loc[group.head(5).index, "ReputationGroup"] = f"Top 5 - {resort_type}"
        # Middle 5 for this type
        mid = len(group) // 2
        df.loc[group.iloc[mid-2 : mid+3].index, "ReputationGroup"] = f"Middle 5 - {resort_type}"
        # Bottom 5 for this type
        df.loc[group.tail(5).index, "ReputationGroup"] = f"Bottom 5 - {resort_type}"

df.to_csv("Data/kenya_resorts_final.csv", index=False)
print(df["ReputationGroup"].value_counts())

ReputationGroup
Middle                     123
Bottom 5 - Marine Coast      5
Top 5 - Marine Coast         5
Middle 5 - Marine Coast      5
Top 5 - Inland Lake          5
Middle 5 - Inland Lake       5
Bottom 5 - Inland Lake       5
Bottom 5 - Safari River      5
Top 5 - Safari River         5
Middle 5 - Safari River      5
Name: count, dtype: int64


**Making the topic columns numeric:** Converting the True/False topic columns to 1/0 so Tableau can add them up.

In [19]:
# Convert the True/False topic columns to 1/0 numbers
mention_cols = [c for c in df.columns if c.startswith("Mentions_")]
df[mention_cols] = df[mention_cols].astype(int)

# Save the master dataset
df.to_csv("Data/kenya_resorts_master.csv", index=False)

# Check: should print Service 159, Food 148, Rooms 145, Setting 129, Cleanliness 112, Noise 94, Price 66
print(df[mention_cols].sum().sort_values(ascending=False))
print(df.shape)

Mentions_Service            159
Mentions_Food               148
Mentions_Rooms              145
Mentions_Setting & views    129
Mentions_Cleanliness        112
Mentions_Noise               94
Mentions_Price               66
dtype: int64
(168, 24)


### **SECTION G: ANALYSIS**

**Loading the master dataset:** Checking that it has 168 resorts, 12 columns and no repeated column names.

In [22]:
# Load the master dataset
df = pd.read_csv("Data/kenya_resorts_master.csv")

# Check it: number of rows and columns
print(df.shape)

# Check it: are any column names repeated? (should be False)
print(df.columns.duplicated().any())

# Check it: the highest worship count (31 = today's recount, 32 = yesterday's file)
print(df["WorshipCount"].max())

(168, 12)
False
31


**Noise tiers:** Sorting resorts into Quiet (0 to 3 noise sources), Medium (4 to 14) and Noisy (15 or more). This gives 60, 54 and 54 resorts.

In [23]:
# Sort each resort into a noise tier, using the same cut-offs as Tableau
def noise_tier(total):
    if total <= 3:
        return "1 Quiet (0-3)"
    elif total <= 14:
        return "2 Medium (4-14)"
    else:
        return "3 Noisy (15+)"

# Apply it to every row and store the result in a new column
df["NoiseTier"] = df["TotalNoiseSources"].apply(noise_tier)

# Check: how many resorts are in each tier
print(df["NoiseTier"].value_counts().sort_index())

NoiseTier
1 Quiet (0-3)      60
2 Medium (4-14)    54
3 Noisy (15+)      54
Name: count, dtype: int64


**Ratings by noise tier:** The average Google rating for each water body type and noise tier. At inland lakes, ratings step down as noise rises. At the coast, they do not.

In [24]:
# Average rating for each water body type and noise tier
result = df.groupby(["WaterBodyType", "NoiseTier"])["GoogleRating"].agg(["count", "mean"]).round(2)

print(result)

                               count  mean
WaterBodyType NoiseTier                   
Inland Lake   1 Quiet (0-3)       23  4.45
              2 Medium (4-14)     26  4.30
              3 Noisy (15+)       30  4.14
Marine Coast  1 Quiet (0-3)       16  4.38
              2 Medium (4-14)     28  4.49
              3 Noisy (15+)       23  4.49
Safari River  1 Quiet (0-3)       21  4.54


### **SECTION H: CONCLUSIONS**

#### **Dimension 1: Reputation**
* **Insight:** The dashboard allows users to explore the Top 5, Middle 5, and Bottom 5 resorts for each specific resort type (Coast, Lake, Safari).

#### **Dimension 2: The Drivers**
* **Insight:** Guests care most about Service and Food. Noise is mentioned less frequently overall, but it remains a critical pain point for specific locations.

#### **Dimension 3: The Pain Point**
* **Insight:** Noisier resorts get significantly lower ratings at inland lakes, but not at the coast. The context is that lake travelers expect peace, while coast travelers expect lively surroundings.

#### **Dimension 4: Destination**
* **Insight:** An interactive map allows users to click on a resort type or specific resort to filter the entire dashboard.

**Limits:** This shows a correlation, not a cause. A zero noise count means "no recorded noise nearby," not "silent."

### **SECTION I: SENTIMENT ANALYSIS**

**The Flaw:** The previous keyword analysis counted *mentions* but couldn't tell if the review was positive or negative. "The food was terrible" still counted as a Food mention.

**The Fix:** To make the Priority filter truly useful, I used VADER (Valence Aware Dictionary and sEntiment Reasoner) to score every review text. 
* A score > 0.05 means the review was positive.
* A score < -0.05 means the review was negative.

This new `Sentiment_Score` column allows Tableau to filter for *positive* experiences only.

In [3]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load your final data
df = pd.read_csv("Data/kenya_resorts_final.csv")

# Initialize the analyzer
analyzer = SentimentIntensityAnalyzer()

# Create a function to get the compound score
def get_sentiment(text):
    if pd.isna(text) or text == "":
        return 0.0
    return analyzer.polarity_scores(str(text))['compound']

# Apply it to the ReviewText column
print("Analyzing sentiment...")
df['Sentiment_Score'] = df['ReviewText'].apply(get_sentiment)

# Save it back to the CSV
df.to_csv("Data/kenya_resorts_final.csv", index=False)

print("Done! Here is a sample:")
print(df[['ResortName', 'Sentiment_Score']].head())

Analyzing sentiment...
Done! Here is a sample:
                             ResortName  Sentiment_Score
0         Diani Reef Beach Resort & Spa           0.9994
1                  Swahili Beach Resort           0.9998
2  Diamonds Leisure Beach & Golf Resort           0.9997
3                    Coral Beach Resort           0.9992
4           Southern Palms Beach Resort           0.9995
